# M1 — MELD in the Hi-EF format: MCIS windows, annotation file and Hi-EF-style clip features

* **MCIS**: three consecutive utterances I, II, III of one MELD dialogue, then utterance IV spoken by a **different**
  speaker than III (Hi-EF's interaction rule). Label = MELD emotion of IV, mapped to the Hi-EF names
  (anger→angry, joy→happy, sadness→sad). Splits: MELD train → `train`, dev → `val` (early stopping), test → `test`.
  `source_folder` = dialogue. Speaker names are **not** written anywhere the models read.
* **Annotation file** in the Hi-EF layout (`MELD/Hi-EF/annotation.csv`, no header: 0 clip id, 1 text, 5 polarity =
  MELD sentiment, 7 emotion, 8 uncertainty = 1). Clip id = `<tr|dv|te><dialogue:05d>/<utterance:03d>`.
* **Features per clip I–III** in the `hi-ef-features-v2` format (re-implemented; the original Hi-EF extraction script
  is not available): 16 evenly spaced frames; CLIP ViT-B/32 image features of the largest detected face per frame
  (`face_features`, `face_valid_mask`) and of the whole frame (`ori_features`); CLIP text features of the transcript;
  AudioCLIP ESResNeXt-FBSP AudioSet outputs (527) on 3 s of de-silenced audio at 22.05 kHz, as in the Hi-EF code.
* Clip IV is never processed. Output: `meld_split.csv`, `meld_hief/`, `meld_features/` → make it a dataset for M2/M3.

Needs Internet (MELD.Raw ≈ 10 GB, unpacked in `/tmp`) or an attached copy of MELD.Raw (`MELD_LOCAL`).

In [ ]:
!pip install -q insightface onnx termcolor
!pip uninstall -y -q onnxruntime onnxruntime-gpu
!pip install -q "onnxruntime-gpu==1.22.0"

In [ ]:
# ======== CONFIG ========
OUT_DIR = "/kaggle/working"
WORK = "/tmp/meld"                  # large scratch space for the raw videos
MELD_LOCAL = None                   # or the path of an attached dataset holding the MELD.Raw videos and CSVs
ESR_URL = "https://github.com/AndreyGuzhov/AudioCLIP/releases/download/v0.1/ESRNXFBSP.pt"
N_FRAMES, DET_SIZE, MIN_DET_SCORE, MIN_FACE_PX, FACE_MARGIN = 16, (640, 640), 0.5, 24, 0.2
AUDIO_SR = 22050                    # AudioCLIP sample rate (the Hi-EF preprocessing)
DEBUG_N = None                      # e.g. 40 windows for a smoke test

In [ ]:
import os, re, glob, subprocess
from concurrent.futures import ThreadPoolExecutor

MELD_URL = "https://web.eecs.umich.edu/~mihalcea/downloads/MELD.Raw.tar.gz"
CSV_URL = "https://raw.githubusercontent.com/declare-lab/MELD/master/data/MELD/{}_sent_emo.csv"
EMO_MAP = {'anger': 'angry', 'disgust': 'disgust', 'fear': 'fear', 'joy': 'happy', 'neutral': 'neutral',
           'sadness': 'sad', 'surprise': 'surprise'}
SPLITS = (('tr', 'train', 'train'), ('dv', 'dev', 'val'), ('te', 'test', 'test'))   # prefix, MELD name, Hi-EF name


def fetch_meld():
    # returns {(prefix, dialogue, utterance): mp4 path} and {prefix: csv path}
    root = MELD_LOCAL
    if not root:
        root = WORK
        os.makedirs(WORK, exist_ok=True)
        if not glob.glob(f"{WORK}/**/*.mp4", recursive=True):
            print("downloading and unpacking MELD.Raw (about 10 GB) ...", flush=True)
            subprocess.run(f"wget -q -O - {MELD_URL} | tar -xz -C {WORK}", shell=True, check=True)
            for inner in glob.glob(f"{WORK}/**/*.tar.gz", recursive=True):
                subprocess.run(['tar', '-xzf', inner, '-C', os.path.dirname(inner)], check=True)
                os.remove(inner)
    vids = {}
    for p in glob.glob(f"{root}/**/*.mp4", recursive=True):
        m = re.fullmatch(r'dia(\d+)_utt(\d+)\.mp4', os.path.basename(p))
        if not m:
            continue                                   # skips macOS '._' files and other names
        low = os.path.relpath(p, root).lower()
        s = 'te' if 'test' in low else 'dv' if 'dev' in low else 'tr' if 'train' in low else None
        if s:
            vids[(s, int(m[1]), int(m[2]))] = p
    csvs = {}
    for s, name, _ in SPLITS:
        found = glob.glob(f"{root}/**/{name}_sent_emo.csv", recursive=True)
        if found:
            csvs[s] = found[0]
        else:
            csvs[s] = f"{WORK}/{name}_sent_emo.csv"
            os.makedirs(WORK, exist_ok=True)
            subprocess.run(['wget', '-q', '-O', csvs[s], CSV_URL.format(name)], check=True)
    print("videos found per split:", {s: sum(k[0] == s for k in vids) for s, _, _ in SPLITS})
    return vids, csvs


def clip_id(s, d, u):
    return f"{s}{int(d):05d}/{int(u):03d}"


def extract_wavs(clips, vids, sr):
    # mono wav per clip at <WORK>/audio/<ep>/<num>.wav (ffmpeg), skipped when present
    def one(c):
        s, d, u = c[:2], int(c[2:7]), int(c[8:])
        out = f"{WORK}/audio/{c}.wav"
        if os.path.exists(out) or (s, d, u) not in vids:
            return
        os.makedirs(os.path.dirname(out), exist_ok=True)
        subprocess.run(['ffmpeg', '-loglevel', 'error', '-y', '-i', vids[(s, d, u)], '-vn', '-ac', '1', '-ar', str(sr),
                        out], check=False)
    with ThreadPoolExecutor(8) as pool:
        list(pool.map(one, clips))
    print("wav files:", len(glob.glob(f"{WORK}/audio/**/*.wav", recursive=True)), flush=True)

import numpy as np, pandas as pd
vids, csvs = fetch_meld()

## MCIS windows, split file and annotation file

In [ ]:
U = []
for s, name, hname in SPLITS:
    d = pd.read_csv(csvs[s])
    d['s'], d['hsplit'] = s, hname
    U.append(d)
U = pd.concat(U, ignore_index=True)
U['cid'] = [clip_id(s, d, u) for s, d, u in zip(U.s, U.Dialogue_ID, U.Utterance_ID)]
U['emo'] = U.Emotion.map(EMO_MAP)
assert U.emo.notna().all()
U['text'] = U.Utterance.astype(str).str.replace('\x92', "'").str.replace('\x91', "'").str.replace('\x93', '"').str.replace('\x94', '"')
rows = []
for (s, d), g in U.sort_values(['s', 'Dialogue_ID', 'Utterance_ID']).groupby(['s', 'Dialogue_ID'], sort=False):
    c, sp_, em = g.cid.tolist(), g.Speaker.tolist(), g.emo.tolist()
    for i in range(3, len(g)):
        if sp_[i] == sp_[i - 1]:
            continue                                   # Hi-EF: the last two clips feature different people
        rows.append({'split': g.hsplit.iloc[0], 'source_folder': f"{s}{int(d):05d}", 'clip1': c[i - 3],
                     'clip2': c[i - 2], 'clip3': c[i - 1], 'clip4': c[i], 'clip3_emotion': em[i - 1],
                     'clip4_emotion': em[i]})
SP = pd.DataFrame(rows)
if DEBUG_N:
    SP = SP.groupby('split', group_keys=False).head(DEBUG_N)
SP.insert(0, 'sample_id', [f"meld{n:05d}" for n in range(len(SP))])
SP.to_csv(f"{OUT_DIR}/meld_split.csv", index=False)
ann_dir = f"{OUT_DIR}/meld_hief/MELD/Hi-EF"
os.makedirs(ann_dir, exist_ok=True)
A = pd.DataFrame({0: U.cid, 1: U.text, 2: '', 3: '', 4: '', 5: U.Sentiment.str.lower(), 6: '', 7: U.emo, 8: '1'})
A.to_csv(f"{ann_dir}/annotation.csv", header=False, index=False)
print("MCIS per split:", SP.split.value_counts().to_dict())
print("label share:", SP.clip4_emotion.value_counts(normalize=True).round(3).to_dict())
print(f"P(clip IV emotion == clip III emotion) = {(SP.clip4_emotion == SP.clip3_emotion).mean():.3f}")
CLIPS = sorted(set(SP[['clip1', 'clip2', 'clip3']].values.ravel()))
miss = [c for c in CLIPS if (c[:2], int(c[2:7]), int(c[8:])) not in vids]
print(f"clips I-III {len(CLIPS)} | without video {len(miss)} (features stay empty for those)")

## Audio, models and per-clip features

In [ ]:
import torch, cv2, sys, urllib.request
from PIL import Image
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
extract_wavs(CLIPS, vids, AUDIO_SR)

# AudioCLIP's ESResNeXt-FBSP (code as in this repository's esresnet/, with minimal stubs for its imports)
PKG = f"{WORK}/esr_pkg"
for rel, src in {'esresnet/__init__.py': "from .base import ESResNet\nfrom .base import ESResNeXt\nfrom .fbsp import ESResNetFBSP\nfrom .fbsp import ESResNeXtFBSP\nfrom .attention import Attention2d\n\n\n__all__ = ['ESResNet', 'ESResNeXt', 'ESResNetFBSP', 'ESResNeXtFBSP', 'Attention2d']\n", 'esresnet/attention.py': 'import torch\nimport torch.nn.functional as F\n\nfrom typing import Tuple\n\n\nclass Attention2d(torch.nn.Module):\n\n    def __init__(self,\n                 in_channels: int,\n                 out_channels: int,\n                 num_kernels: int,\n                 kernel_size: Tuple[int, int],\n                 padding_size: Tuple[int, int]):\n\n        super(Attention2d, self).__init__()\n\n        self.conv_depth = torch.nn.Conv2d(\n            in_channels=in_channels,\n            out_channels=in_channels * num_kernels,\n            kernel_size=kernel_size,\n            padding=padding_size,\n            groups=in_channels\n        )\n        self.conv_point = torch.nn.Conv2d(\n            in_channels=in_channels * num_kernels,\n            out_channels=out_channels,\n            kernel_size=(1, 1)\n        )\n        self.bn = torch.nn.BatchNorm2d(num_features=out_channels)\n        self.activation = torch.nn.Sigmoid()\n\n    def forward(self, x: torch.Tensor, size: torch.Size) -> torch.Tensor:\n        x = F.adaptive_max_pool2d(x, size)\n        x = self.conv_depth(x)\n        x = self.conv_point(x)\n        x = self.bn(x)\n        x = self.activation(x)\n\n        return x\n', 'esresnet/base.py': 'import termcolor\n\nimport numpy as np\nimport scipy.signal as sps\n\nimport torch\nimport torch.nn.functional as F\n\nimport torchvision as tv\n\nimport ignite_trainer as it\n\nfrom esresnet import attention\nfrom utils.transforms import scale\n\nfrom typing import cast\nfrom typing import List\nfrom typing import Type\nfrom typing import Tuple\nfrom typing import Union\nfrom typing import Optional\n\n\ndef conv3x3(in_planes: int, out_planes: int, stride=1, groups: int = 1, dilation: Union[int, Tuple[int, int]] = 1):\n    """\n    CREDITS: https://github.com/pytorch/vision\n    3x3 convolution with padding\n    """\n    return torch.nn.Conv2d(\n        in_channels=in_planes,\n        out_channels=out_planes,\n        kernel_size=3,\n        stride=stride,\n        padding=dilation,\n        groups=groups,\n        bias=False,\n        dilation=dilation\n    )\n\n\ndef conv1x1(in_planes: int, out_planes: int, stride: Union[int, Tuple[int, int]] = 1):\n    """\n    CREDITS: https://github.com/pytorch/vision\n    1x1 convolution\n    """\n    return torch.nn.Conv2d(\n        in_channels=in_planes,\n        out_channels=out_planes,\n        kernel_size=1,\n        stride=stride,\n        bias=False\n    )\n\n\nclass BasicBlock(torch.nn.Module):\n\n    """\n    CREDITS: https://github.com/pytorch/vision\n    """\n\n    expansion: int = 1\n\n    def __init__(self,\n                 inplanes: int,\n                 planes: int,\n                 stride: Union[int, Tuple[int, int]] = 1,\n                 downsample: Optional[torch.nn.Module] = None,\n                 groups: int = 1,\n                 base_width: int = 64,\n                 dilation: Union[int, Tuple[int, int]] = 1,\n                 norm_layer: Optional[Type[torch.nn.Module]] = None):\n\n        super(BasicBlock, self).__init__()\n\n        if norm_layer is None:\n            norm_layer = torch.nn.BatchNorm2d\n        if groups != 1 or base_width != 64:\n            raise ValueError(\'BasicBlock only supports groups=1 and base_width=64\')\n        if dilation > 1:\n            raise NotImplementedError("Dilation > 1 not supported in BasicBlock")\n\n        # Both self.conv1 and self.downsample layers downsample the input when stride != 1\n        self.conv1 = conv3x3(inplanes, planes, stride)\n        self.bn1 = norm_layer(planes)\n        self.relu = torch.nn.ReLU()\n        self.conv2 = conv3x3(planes, planes)\n        self.bn2 = norm_layer(planes)\n        self.downsample = downsample\n        self.stride = stride\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        identity = x\n\n        out = self.conv1(x)\n        out = self.bn1(out)\n        out = self.relu(out)\n\n        out = self.conv2(out)\n        out = self.bn2(out)\n\n        if self.downsample is not None:\n            identity = self.downsample(x)\n\n        out += identity\n        out = self.relu(out)\n\n        return out\n\n\nclass Bottleneck(torch.nn.Module):\n\n    """\n    CREDITS: https://github.com/pytorch/vision\n    """\n\n    expansion: int = 4\n\n    def __init__(self,\n                 inplanes: int,\n                 planes: int,\n                 stride: Union[int, Tuple[int, int]] = 1,\n                 downsample: Optional[torch.nn.Module] = None,\n                 groups: int = 1,\n                 base_width: int = 64,\n                 dilation: Union[int, Tuple[int, int]] = 1,\n                 norm_layer: Optional[Type[torch.nn.Module]] = None):\n\n        super(Bottleneck, self).__init__()\n\n        if norm_layer is None:\n            norm_layer = torch.nn.BatchNorm2d\n\n        width = int(planes * (base_width / 64.0)) * groups\n\n        self.conv1 = conv1x1(inplanes, width)\n        self.bn1 = norm_layer(width)\n        self.conv2 = conv3x3(width, width, stride, groups, dilation)\n        self.bn2 = norm_layer(width)\n        self.conv3 = conv1x1(width, planes * self.expansion)\n        self.bn3 = norm_layer(planes * self.expansion)\n        self.relu = torch.nn.ReLU()\n        self.downsample = downsample\n        self.stride = stride\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        identity = x\n\n        out = self.conv1(x)\n        out = self.bn1(out)\n        out = self.relu(out)\n\n        out = self.conv2(out)\n        out = self.bn2(out)\n        out = self.relu(out)\n\n        out = self.conv3(out)\n        out = self.bn3(out)\n\n        if self.downsample is not None:\n            identity = self.downsample(x)\n\n        out += identity\n        out = self.relu(out)\n\n        return out\n\n\nclass ResNetWithAttention(it.AbstractNet):\n\n    """\n    CREDITS: https://github.com/pytorch/vision\n    """\n\n    def __init__(self,\n                 block: Type[Union[BasicBlock, Bottleneck]],\n                 layers: List[int],\n                 apply_attention: bool = False,\n                 num_channels: int = 3,\n                 num_classes: int = 1000,\n                 zero_init_residual: bool = False,\n                 groups: int = 1,\n                 width_per_group: int = 64,\n                 replace_stride_with_dilation: bool = None,\n                 norm_layer: Optional[Type[torch.nn.Module]] = None):\n\n        super(ResNetWithAttention, self).__init__()\n\n        self.apply_attention = apply_attention\n\n        if norm_layer is None:\n            norm_layer = torch.nn.BatchNorm2d\n\n        self._norm_layer = norm_layer\n\n        self.inplanes = 64\n        self.dilation = 1\n\n        if replace_stride_with_dilation is None:\n            replace_stride_with_dilation = [False, False, False]\n\n        if len(replace_stride_with_dilation) != 3:\n            raise ValueError(\n                f\'replace_stride_with_dilation should be None or a 3-element tuple, got {replace_stride_with_dilation}\'\n            )\n\n        self.groups = groups\n        self.base_width = width_per_group\n\n        self.conv1 = torch.nn.Conv2d(num_channels, self.inplanes, kernel_size=7, stride=2, padding=3, bias=False)\n        self.bn1 = norm_layer(self.inplanes)\n        self.relu = torch.nn.ReLU()\n        self.maxpool = torch.nn.MaxPool2d(kernel_size=3, stride=2, padding=1)\n\n        self.layer1 = self._make_layer(block, 64, layers[0])\n        if self.apply_attention:\n            self.att1 = attention.Attention2d(\n                in_channels=64,\n                out_channels=64 * block.expansion,\n                num_kernels=1,\n                kernel_size=(3, 1),\n                padding_size=(1, 0)\n            )\n\n        self.layer2 = self._make_layer(block, 128, layers[1], stride=2, dilate=replace_stride_with_dilation[0])\n        if self.apply_attention:\n            self.att2 = attention.Attention2d(\n                in_channels=64 * block.expansion,\n                out_channels=128 * block.expansion,\n                num_kernels=1,\n                kernel_size=(1, 5),\n                padding_size=(0, 2)\n            )\n\n        self.layer3 = self._make_layer(block, 256, layers[2], stride=2, dilate=replace_stride_with_dilation[1])\n        if self.apply_attention:\n            self.att3 = attention.Attention2d(\n                in_channels=128 * block.expansion,\n                out_channels=256 * block.expansion,\n                num_kernels=1,\n                kernel_size=(3, 1),\n                padding_size=(1, 0)\n            )\n\n        self.layer4 = self._make_layer(block, 512, layers[3], stride=2, dilate=replace_stride_with_dilation[2])\n        if self.apply_attention:\n            self.att4 = attention.Attention2d(\n                in_channels=256 * block.expansion,\n                out_channels=512 * block.expansion,\n                num_kernels=1,\n                kernel_size=(1, 5),\n                padding_size=(0, 2)\n            )\n\n        self.avgpool = torch.nn.AdaptiveAvgPool2d((1, 1))\n        if self.apply_attention:\n            self.att5 = attention.Attention2d(\n                in_channels=512 * block.expansion,\n                out_channels=512 * block.expansion,\n                num_kernels=1,\n                kernel_size=(3, 5),\n                padding_size=(1, 2)\n            )\n\n        self.fc = torch.nn.Linear(512 * block.expansion, num_classes)\n\n        for m in self.modules():\n            if isinstance(m, torch.nn.Conv2d):\n                torch.nn.init.kaiming_normal_(m.weight, mode=\'fan_out\', nonlinearity=\'relu\')\n            elif isinstance(m, (torch.nn.BatchNorm2d, torch.nn.GroupNorm)):\n                torch.nn.init.constant_(m.weight, 1)\n                torch.nn.init.constant_(m.bias, 0)\n\n        if zero_init_residual:\n            for m in self.modules():\n                if isinstance(m, Bottleneck):\n                    torch.nn.init.constant_(m.bn3.weight, 0)\n                elif isinstance(m, BasicBlock):\n                    torch.nn.init.constant_(m.bn2.weight, 0)\n\n    def _make_layer(self,\n                    block: Type[Union[BasicBlock, Bottleneck]],\n                    planes: int,\n                    blocks: int,\n                    stride: Union[int, Tuple[int, int]] = 1,\n                    dilate: bool = False) -> torch.nn.Module:\n\n        norm_layer = self._norm_layer\n        downsample = None\n        previous_dilation = self.dilation\n\n        if dilate:\n            self.dilation *= stride\n            stride = 1\n\n        if stride != 1 or self.inplanes != planes * block.expansion:\n            downsample = torch.nn.Sequential(\n                conv1x1(self.inplanes, planes * block.expansion, stride),\n                norm_layer(planes * block.expansion)\n            )\n\n        layers = list()\n        layers.append(block(\n            self.inplanes,\n            planes,\n            stride,\n            downsample,\n            self.groups,\n            self.base_width,\n            previous_dilation,\n            norm_layer\n        ))\n        self.inplanes = planes * block.expansion\n        for _ in range(1, blocks):\n            layers.append(block(\n                self.inplanes,\n                planes,\n                groups=self.groups,\n                base_width=self.base_width,\n                dilation=self.dilation,\n                norm_layer=norm_layer\n            ))\n\n        return torch.nn.Sequential(*layers)\n\n    def _forward_pre_processing(self, x: torch.Tensor) -> torch.Tensor:\n        x = x.to(torch.get_default_dtype())\n\n        return x\n\n    def _forward_pre_features(self, x: torch.Tensor) -> torch.Tensor:\n        x = self.conv1(x)\n        x = self.bn1(x)\n        x = self.relu(x)\n        x = self.maxpool(x)\n\n        return x\n\n    def _forward_features(self, x: torch.Tensor) -> torch.Tensor:\n        x = self._forward_pre_features(x)\n\n        if self.apply_attention:\n            x_att = x.clone()\n            x = self.layer1(x)\n            x_att = self.att1(x_att, x.shape[-2:])\n            x = x * x_att\n\n            x_att = x.clone()\n            x = self.layer2(x)\n            x_att = self.att2(x_att, x.shape[-2:])\n            x = x * x_att\n\n            x_att = x.clone()\n            x = self.layer3(x)\n            x_att = self.att3(x_att, x.shape[-2:])\n            x = x * x_att\n\n            x_att = x.clone()\n            x = self.layer4(x)\n            x_att = self.att4(x_att, x.shape[-2:])\n            x = x * x_att\n        else:\n            x = self.layer1(x)\n            x = self.layer2(x)\n            x = self.layer3(x)\n            x = self.layer4(x)\n\n        return x\n\n    def _forward_reduction(self, x: torch.Tensor) -> torch.Tensor:\n        if self.apply_attention:\n            x_att = x.clone()\n            x = self.avgpool(x)\n            x_att = self.att5(x_att, x.shape[-2:])\n            x = x * x_att\n        else:\n            x = self.avgpool(x)\n\n        x = torch.flatten(x, 1)\n\n        return x\n\n    def _forward_classifier(self, x: torch.Tensor) -> torch.Tensor:\n        x = self.fc(x)\n\n        return x\n\n    def forward(self,\n                x: torch.Tensor,\n                y: Optional[torch.Tensor] = None) -> Union[torch.Tensor, Tuple[torch.Tensor, torch.Tensor]]:\n\n        x = self._forward_pre_processing(x)\n        x = self._forward_features(x)\n        x = self._forward_reduction(x)\n        y_pred = self._forward_classifier(x)\n\n        loss = None\n        if y is not None:\n            loss = self.loss_fn(y_pred, y).mean()\n\n        return y_pred if loss is None else (y_pred, loss)\n\n    def loss_fn(self, y_pred: torch.Tensor, y: torch.Tensor) -> torch.Tensor:\n        if isinstance(y_pred, tuple):\n            y_pred, *_ = y_pred\n\n        if y_pred.shape == y.shape:\n            loss_pred = F.binary_cross_entropy_with_logits(\n                y_pred,\n                y.to(dtype=y_pred.dtype, device=y_pred.device),\n                reduction=\'sum\'\n            ) / y_pred.shape[0]\n        else:\n            loss_pred = F.cross_entropy(y_pred, y.to(y_pred.device))\n\n        return loss_pred\n\n    @property\n    def loss_fn_name(self) -> str:\n        return \'Cross Entropy\'\n\n\nclass _ESResNet(ResNetWithAttention):\n\n    @staticmethod\n    def loading_function(*args, **kwargs) -> torch.nn.Module:\n        raise NotImplementedError\n\n    def __init__(self,\n                 block: Type[Union[BasicBlock, Bottleneck]],\n                 layers: List[int],\n                 apply_attention: bool = False,\n                 n_fft: int = 256,\n                 hop_length: Optional[int] = None,\n                 win_length: Optional[int] = None,\n                 window: Optional[str] = None,\n                 normalized: bool = False,\n                 onesided: bool = True,\n                 spec_height: int = 224,\n                 spec_width: int = 224,\n                 num_classes: int = 1000,\n                 pretrained: Union[bool, str] = False,\n                 lock_pretrained: Optional[Union[bool, List[str]]] = None,\n                 zero_init_residual: bool = False,\n                 groups: int = 1,\n                 width_per_group: int = 64,\n                 replace_stride_with_dilation: bool = None,\n                 norm_layer: Optional[Type[torch.nn.Module]] = None):\n\n        super(_ESResNet, self).__init__(\n            block=block,\n            layers=layers,\n            apply_attention=apply_attention,\n            num_channels=3,\n            num_classes=num_classes,\n            zero_init_residual=zero_init_residual,\n            groups=groups,\n            width_per_group=width_per_group,\n            replace_stride_with_dilation=replace_stride_with_dilation,\n            norm_layer=norm_layer\n        )\n\n        self.num_classes = num_classes\n\n        self.fc = torch.nn.Linear(\n            in_features=self.fc.in_features,\n            out_features=self.num_classes,\n            bias=self.fc.bias is not None\n        )\n\n        if hop_length is None:\n            hop_length = int(np.floor(n_fft / 4))\n\n        if win_length is None:\n            win_length = n_fft\n\n        if window is None:\n            window = \'boxcar\'\n\n        self.n_fft = n_fft\n        self.win_length = win_length\n        self.hop_length = hop_length\n\n        self.normalized = normalized\n        self.onesided = onesided\n\n        self.spec_height = spec_height\n        self.spec_width = spec_width\n\n        self.pretrained = pretrained\n        self._inject_members()\n        if pretrained:\n            err_msg = self.load_pretrained()\n\n            unlocked_weights = list()\n\n            for name, p in self.named_parameters():\n                unlock = True\n                if isinstance(lock_pretrained, bool):\n                    if lock_pretrained and name not in err_msg:\n                        unlock = False\n                elif isinstance(lock_pretrained, list):\n                    if name in lock_pretrained:\n                        unlock = False\n\n                p.requires_grad_(unlock)\n                if unlock:\n                    unlocked_weights.append(name)\n\n            print(f\'Following weights are unlocked: {unlocked_weights}\')\n\n        window_buffer: torch.Tensor = torch.from_numpy(\n            sps.get_window(window=window, Nx=win_length, fftbins=True)\n        ).to(torch.get_default_dtype())\n        self.register_buffer(\'window\', window_buffer)\n\n        self.log10_eps = 1e-18\n\n        if self.apply_attention and pretrained and not isinstance(pretrained, str):\n            self._reset_attention()\n\n    def _inject_members(self):\n        pass\n\n    def _reset_attention(self):\n        print(termcolor.colored(\'Resetting attention blocks\', \'green\'))\n\n        self.att1.bn.weight.data.fill_(1.0)\n        self.att1.bn.bias.data.fill_(1.0)\n\n        self.att2.bn.weight.data.fill_(1.0)\n        self.att2.bn.bias.data.fill_(1.0)\n\n        self.att3.bn.weight.data.fill_(1.0)\n        self.att3.bn.bias.data.fill_(1.0)\n\n        self.att4.bn.weight.data.fill_(1.0)\n        self.att4.bn.bias.data.fill_(1.0)\n\n        self.att5.bn.weight.data.fill_(1.0)\n        self.att5.bn.bias.data.fill_(1.0)\n\n    def load_pretrained(self) -> str:\n        if isinstance(self.pretrained, bool):\n            state_dict = self.loading_func(pretrained=True).state_dict()\n        else:\n            state_dict = torch.load(self.pretrained, map_location=\'cpu\')\n\n        err_msg = \'\'\n        try:\n            self.load_state_dict(state_dict=state_dict, strict=True)\n        except RuntimeError as ex:\n            err_msg += f\'While loading some errors occurred.\\n{ex}\'\n            print(termcolor.colored(err_msg, \'red\'))\n\n        return err_msg\n\n    def spectrogram(self, x: torch.Tensor) -> torch.Tensor:\n        spec = torch.stft(\n            x.view(-1, x.shape[-1]),\n            n_fft=self.n_fft,\n            hop_length=self.hop_length,\n            win_length=self.win_length,\n            window=self.window,\n            pad_mode=\'reflect\',\n            normalized=self.normalized,\n            onesided=True\n        )\n\n        if not self.onesided:\n            spec = torch.cat((torch.flip(spec, dims=(-3,)), spec), dim=-3)\n\n        return spec\n\n    def split_spectrogram(self, spec: torch.Tensor, batch_size: int) -> torch.Tensor:\n        spec_height_per_band = spec.shape[-3] // self.conv1.in_channels\n        spec_height_single_band = self.conv1.in_channels * spec_height_per_band\n        spec = spec[:, :spec_height_single_band]\n\n        spec = spec.reshape(batch_size, -1, spec.shape[-3] // self.conv1.in_channels, *spec.shape[-2:])\n\n        return spec\n\n    def spectrogram_to_power(self, spec: torch.Tensor) -> torch.Tensor:\n        spec_height = spec.shape[-3] if self.spec_height < 1 else self.spec_height\n        spec_width = spec.shape[-2] if self.spec_width < 1 else self.spec_width\n\n        pow_spec = spec[..., 0] ** 2 + spec[..., 1] ** 2\n\n        if spec_height != pow_spec.shape[-2] or spec_width != pow_spec.shape[-1]:\n            pow_spec = F.interpolate(\n                pow_spec,\n                size=(spec_height, spec_width),\n                mode=\'bilinear\',\n                align_corners=True\n            )\n\n        return pow_spec\n\n    def _forward_pre_processing(self, x: torch.Tensor) -> torch.Tensor:\n        x = super(_ESResNet, self)._forward_pre_processing(x)\n        x = scale(x, -32768.0, 32767, -1.0, 1.0)\n\n        spec = self.spectrogram(x)\n        spec_split_ch = self.split_spectrogram(spec, x.shape[0])\n        pow_spec_split_ch = self.spectrogram_to_power(spec_split_ch)\n        pow_spec_split_ch = torch.where(\n            cast(torch.Tensor, pow_spec_split_ch > 0.0),\n            pow_spec_split_ch,\n            torch.full_like(pow_spec_split_ch, self.log10_eps)\n        )\n        pow_spec_split_ch = pow_spec_split_ch.reshape(\n            x.shape[0], -1, self.conv1.in_channels, *pow_spec_split_ch.shape[-2:]\n        )\n        x_db = torch.log10(pow_spec_split_ch).mul(10.0)\n\n        return x_db\n\n    def _forward_features(self, x_db: torch.Tensor) -> List[torch.Tensor]:\n        outputs = list()\n        for ch_idx in range(x_db.shape[1]):\n            ch = x_db[:, ch_idx]\n            out = super(_ESResNet, self)._forward_features(ch)\n            outputs.append(out)\n\n        return outputs\n\n    def _forward_reduction(self, x: List[torch.Tensor]) -> torch.Tensor:\n        outputs = list()\n        for ch in x:\n            out = super(_ESResNet, self)._forward_reduction(ch)\n            outputs.append(out)\n        outputs = torch.stack(outputs, dim=-1).sum(dim=-1)\n\n        return outputs\n\n\nclass ESResNet(_ESResNet):\n\n    loading_func = staticmethod(tv.models.resnet50)\n\n    def __init__(self,\n                 n_fft: int = 256,\n                 hop_length: Optional[int] = None,\n                 win_length: Optional[int] = None,\n                 window: Optional[str] = None,\n                 normalized: bool = False,\n                 onesided: bool = True,\n                 spec_height: int = 224,\n                 spec_width: int = 224,\n                 num_classes: int = 1000,\n                 apply_attention: bool = False,\n                 pretrained: bool = False,\n                 lock_pretrained: Optional[Union[bool, List[str]]] = None):\n\n        super(ESResNet, self).__init__(\n            block=Bottleneck,\n            layers=[3, 4, 6, 3],\n            apply_attention=apply_attention,\n            n_fft=n_fft,\n            hop_length=hop_length,\n            win_length=win_length,\n            window=window,\n            normalized=normalized,\n            onesided=onesided,\n            spec_height=spec_height,\n            spec_width=spec_width,\n            num_classes=num_classes,\n            pretrained=pretrained,\n            lock_pretrained=lock_pretrained\n        )\n\n\nclass ESResNeXt(_ESResNet):\n\n    loading_func = staticmethod(tv.models.resnext50_32x4d)\n\n    def __init__(self,\n                 n_fft: int = 256,\n                 hop_length: Optional[int] = None,\n                 win_length: Optional[int] = None,\n                 window: Optional[str] = None,\n                 normalized: bool = False,\n                 onesided: bool = True,\n                 spec_height: int = 224,\n                 spec_width: int = 224,\n                 num_classes: int = 1000,\n                 apply_attention: bool = False,\n                 pretrained: Union[bool, str] = False,\n                 lock_pretrained: Optional[Union[bool, List[str]]] = None):\n\n        super(ESResNeXt, self).__init__(\n            block=Bottleneck,\n            layers=[3, 4, 6, 3],\n            apply_attention=apply_attention,\n            n_fft=n_fft,\n            hop_length=hop_length,\n            win_length=win_length,\n            window=window,\n            normalized=normalized,\n            onesided=onesided,\n            spec_height=spec_height,\n            spec_width=spec_width,\n            num_classes=num_classes,\n            pretrained=pretrained,\n            lock_pretrained=lock_pretrained,\n            groups=32,\n            width_per_group=4\n        )\n', 'esresnet/fbsp.py': "import numpy as np\n\nimport torch\nimport torch.nn.functional as F\n\nimport torchvision as tv\n\nfrom utils import transforms\nfrom esresnet.base import _ESResNet\nfrom esresnet.base import Bottleneck\n\nfrom typing import cast\nfrom typing import List\nfrom typing import Tuple\nfrom typing import Union\nfrom typing import Optional\n\n\nclass LinearFBSP(torch.nn.Module):\n\n    def __init__(self, out_features: int, bias: bool = True, normalized: bool = False):\n        super(LinearFBSP, self).__init__()\n\n        self.out_features = out_features\n        self.normalized = normalized\n        self.eps = 1e-8\n\n        default_dtype = torch.get_default_dtype()\n\n        self.register_parameter('m', torch.nn.Parameter(torch.zeros(self.out_features, dtype=default_dtype)))\n        self.register_parameter('fb', torch.nn.Parameter(torch.ones(self.out_features, dtype=default_dtype)))\n        self.register_parameter('fc', torch.nn.Parameter(torch.arange(self.out_features, dtype=default_dtype)))\n        self.register_parameter(\n            'bias',\n            torch.nn.Parameter(\n                torch.normal(\n                    0.0, 0.5, (self.out_features, 2), dtype=default_dtype\n                ) if bias else cast(\n                    torch.nn.Parameter, None\n                )\n            )\n        )\n\n        self.m.register_hook(lambda grad: grad / (torch.norm(grad, p=float('inf')) + self.eps))\n        self.fb.register_hook(lambda grad: grad / (torch.norm(grad, p=float('inf')) + self.eps))\n        self.fc.register_hook(lambda grad: grad / (torch.norm(grad, p=float('inf')) + self.eps))\n\n    @staticmethod\n    def power(x1: torch.Tensor, x2: torch.Tensor) -> torch.Tensor:\n        magnitudes = (x1[..., 0] ** 2 + x1[..., 1] ** 2) ** 0.5\n        phases = x1[..., 1].atan2(x1[..., 0])\n\n        power_real = x2[..., 0]\n        power_imag = x2[..., 1]\n\n        mag_out = ((magnitudes ** 2) ** (0.5 * power_real) * torch.exp(-power_imag * phases))\n\n        return mag_out.unsqueeze(-1) * torch.stack((\n            (power_real * phases + 0.5 * power_imag * (magnitudes ** 2).log()).cos(),\n            (power_real * phases + 0.5 * power_imag * (magnitudes ** 2).log()).sin()\n        ), dim=-1)\n\n    @staticmethod\n    def sinc(x: torch.Tensor) -> torch.Tensor:\n        return torch.where(cast(torch.Tensor, x == 0), torch.ones_like(x), torch.sin(x) / x)\n\n    def _materialize_weights(self, x: torch.Tensor) -> Tuple[torch.Tensor, bool]:\n        x_is_complex = x.shape[-1] == 2\n        in_features = x.shape[-1 - int(x_is_complex)]\n\n        t = np.pi * torch.linspace(-1.0, 1.0, in_features, dtype=x.dtype, device=x.device).reshape(1, -1, 1) + self.eps\n\n        m = self.m.reshape(-1, 1, 1)\n        fb = self.fb.reshape(-1, 1, 1)\n        fc = self.fc.reshape(-1, 1, 1)\n\n        kernel = torch.cat((torch.cos(fc * t), -torch.sin(fc * t)), dim=-1)  # complex\n        scale = fb.sqrt()  # real\n        win = self.sinc(fb * t / (m + self.eps))  # real\n        win = self.power(\n            torch.cat((win, torch.zeros_like(win)), dim=-1),\n            torch.cat((m, torch.zeros_like(m)), dim=-1)\n        )  # complex\n\n        weights = scale * torch.cat((\n            win[..., :1] * kernel[..., :1] - win[..., 1:] * kernel[..., 1:],\n            win[..., :1] * kernel[..., 1:] + win[..., 1:] * kernel[..., :1]\n        ), dim=-1)\n\n        if self.normalized:\n            weights = weights / (in_features ** 0.5)\n\n        return weights, x_is_complex\n\n    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:\n        weights, x_is_complex = self._materialize_weights(x)\n\n        if x_is_complex:\n            x = torch.stack((\n                F.linear(x[..., 0], weights[..., 0]) - F.linear(x[..., 1], weights[..., 1]),\n                F.linear(x[..., 0], weights[..., 1]) + F.linear(x[..., 1], weights[..., 0])\n            ), dim=-1)\n        else:\n            x = torch.stack((\n                F.linear(x, weights[..., 0]),\n                F.linear(x, weights[..., 1])\n            ), dim=-1)\n\n        if (self.bias is not None) and (self.bias.numel() == (self.out_features * 2)):\n            x = x + self.bias\n\n        return x, weights\n\n    def extra_repr(self) -> str:\n        return 'out_features={}, bias={}, normalized={}'.format(\n            self.out_features,\n            (self.bias is not None) and (self.bias.numel() == (self.out_features * 2)),\n            self.normalized\n        )\n\n\nttf_weights = dict()\n\n\nclass _ESResNetFBSP(_ESResNet):\n\n    def _inject_members(self):\n        self.add_module(\n            'fbsp',\n            LinearFBSP(\n                out_features=int(round(self.n_fft / 2)) + 1 if self.onesided else self.n_fft,\n                normalized=self.normalized,\n                bias=False\n            )\n        )\n\n    def spectrogram(self, x: torch.Tensor) -> torch.Tensor:\n        with torch.no_grad():\n            frames = transforms.frame_signal(\n                signal=x.view(-1, x.shape[-1]),\n                frame_length=self.win_length,\n                hop_length=self.hop_length,\n                window=self.window\n            )\n\n            if self.n_fft > self.win_length:\n                pad_length = self.n_fft - self.win_length\n                pad_left = pad_length // 2\n                pad_right = pad_length - pad_left\n                frames = F.pad(frames, [pad_left, pad_right])\n\n        spec, ttf_weights_ = self.fbsp(frames)\n\n        spec = spec.transpose(-2, -3)\n        ttf_weights[x.device] = ttf_weights_\n\n        return spec\n\n    def loss_ttf(self, device: torch.device) -> torch.Tensor:\n        ttf_norm = torch.norm(ttf_weights[device], p=2, dim=[-1, -2])\n        loss_ttf_norm = F.mse_loss(\n            ttf_norm,\n            torch.full_like(ttf_norm, 1.0 if self.normalized else self.n_fft ** 0.5)\n        )\n\n        return loss_ttf_norm\n\n    def loss_fn(self, y_pred: torch.Tensor, y: torch.Tensor) -> torch.Tensor:\n        loss_pred = super(_ESResNetFBSP, self).loss_fn(y_pred, y)\n        loss_ttf_norm = self.loss_ttf(y_pred.device)\n        loss = loss_pred + loss_ttf_norm\n\n        return loss\n\n\nclass ESResNetFBSP(_ESResNetFBSP):\n\n    loading_func = staticmethod(tv.models.resnet50)\n\n    def __init__(self,\n                 n_fft: int = 256,\n                 hop_length: Optional[int] = None,\n                 win_length: Optional[int] = None,\n                 window: Optional[str] = None,\n                 normalized: bool = False,\n                 onesided: bool = True,\n                 spec_height: int = 224,\n                 spec_width: int = 224,\n                 num_classes: int = 1000,\n                 apply_attention: bool = False,\n                 pretrained: bool = False,\n                 lock_pretrained: Optional[Union[bool, List[str]]] = None):\n\n        super(ESResNetFBSP, self).__init__(\n            block=Bottleneck,\n            layers=[3, 4, 6, 3],\n            apply_attention=apply_attention,\n            n_fft=n_fft,\n            hop_length=hop_length,\n            win_length=win_length,\n            window=window,\n            normalized=normalized,\n            onesided=onesided,\n            spec_height=spec_height,\n            spec_width=spec_width,\n            num_classes=num_classes,\n            pretrained=pretrained,\n            lock_pretrained=lock_pretrained\n        )\n\n\nclass ESResNeXtFBSP(_ESResNetFBSP):\n\n    loading_func = staticmethod(tv.models.resnext50_32x4d)\n\n    def __init__(self,\n                 n_fft: int = 256,\n                 hop_length: Optional[int] = None,\n                 win_length: Optional[int] = None,\n                 window: Optional[str] = None,\n                 normalized: bool = False,\n                 onesided: bool = True,\n                 spec_height: int = 224,\n                 spec_width: int = 224,\n                 num_classes: int = 1000,\n                 apply_attention: bool = False,\n                 pretrained: Union[bool, str] = False,\n                 lock_pretrained: Optional[Union[bool, List[str]]] = None):\n\n        super(ESResNeXtFBSP, self).__init__(\n            block=Bottleneck,\n            layers=[3, 4, 6, 3],\n            apply_attention=apply_attention,\n            n_fft=n_fft,\n            hop_length=hop_length,\n            win_length=win_length,\n            window=window,\n            normalized=normalized,\n            onesided=onesided,\n            spec_height=spec_height,\n            spec_width=spec_width,\n            num_classes=num_classes,\n            pretrained=pretrained,\n            lock_pretrained=lock_pretrained,\n            groups=32,\n            width_per_group=4\n        )\n", 'ignite_trainer/_interfaces.py': "import abc\nimport torch\n\nfrom typing import Tuple\nfrom typing import Union\nfrom typing import Callable\nfrom typing import Optional\n\n\nTensorPair = Tuple[torch.Tensor, torch.Tensor]\nTensorOrTwo = Union[torch.Tensor, TensorPair]\n\n\nclass AbstractNet(abc.ABC, torch.nn.Module):\n\n    @abc.abstractmethod\n    def forward(self, x: torch.Tensor, y: Optional[torch.Tensor] = None) -> TensorOrTwo:\n        pass\n\n    @abc.abstractmethod\n    def loss_fn(self, y_pred: torch.Tensor, y: torch.Tensor) -> torch.Tensor:\n        pass\n\n    @property\n    @abc.abstractmethod\n    def loss_fn_name(self) -> str:\n        pass\n\n\nclass AbstractTransform(abc.ABC, Callable[[torch.Tensor], torch.Tensor]):\n\n    @abc.abstractmethod\n    def __call__(self, x: torch.Tensor) -> torch.Tensor:\n        pass\n\n    def __repr__(self):\n        return self.__class__.__name__ + '()'\n", 'ignite_trainer/__init__.py': 'from ._interfaces import AbstractNet, AbstractTransform\n', 'utils/transforms.py': "import math\nimport numpy as np\nimport torch\n\ndef scale(old_value, old_min, old_max, new_min, new_max):\n    old_range = (old_max - old_min)\n    new_range = (new_max - new_min)\n    new_value = (((old_value - old_min) * new_range) / old_range) + new_min\n\n    return new_value\n\n\ndef frame_signal(signal: torch.Tensor,\n                 frame_length: int,\n                 hop_length: int,\n                 window: torch.Tensor = None) -> torch.Tensor:\n\n    if window is None:\n        window = torch.ones(frame_length, dtype=signal.dtype, device=signal.device)\n\n    if window.shape[0] != frame_length:\n        raise ValueError('Wrong `window` length: expected {}, got {}'.format(window.shape[0], frame_length))\n\n    signal_length = signal.shape[-1]\n\n    if signal_length <= frame_length:\n        num_frames = 1\n    else:\n        num_frames = 1 + int(math.ceil((1.0 * signal_length - frame_length) / hop_length))\n\n    pad_len = int((num_frames - 1) * hop_length + frame_length)\n    if pad_len > signal_length:\n        zeros = torch.zeros(pad_len - signal_length, device=signal.device, dtype=signal.dtype)\n\n        while zeros.dim() < signal.dim():\n            zeros.unsqueeze_(0)\n\n        pad_signal = torch.cat((zeros.expand(*signal.shape[:-1], -1)[..., :zeros.shape[-1] // 2], signal), dim=-1)\n        pad_signal = torch.cat((pad_signal, zeros.expand(*signal.shape[:-1], -1)[..., zeros.shape[-1] // 2:]), dim=-1)\n    else:\n        pad_signal = signal\n\n    indices = torch.arange(0, frame_length, device=signal.device).repeat(num_frames, 1)\n    indices += torch.arange(\n        0,\n        num_frames * hop_length,\n        hop_length,\n        device=signal.device\n    ).repeat(frame_length, 1).t_()\n    indices = indices.long()\n\n    frames = pad_signal[..., indices]\n    frames = frames * window\n\n    return frames\n\n\n", 'utils/__init__.py': ''}.items():
    os.makedirs(os.path.dirname(f"{PKG}/{rel}"), exist_ok=True)
    open(f"{PKG}/{rel}", 'w').write(src)
sys.path.insert(0, PKG)
from esresnet import ESResNeXtFBSP
esr_path = f"{WORK}/ESRNXFBSP.pt"
if not os.path.exists(esr_path):
    urllib.request.urlretrieve(ESR_URL, esr_path)
esr = ESResNeXtFBSP(n_fft=2048, hop_length=561, win_length=1654, window='blackmanharris', normalized=True,
                    onesided=True, spec_height=-1, spec_width=-1, num_classes=527, apply_attention=True, pretrained=False)
r = esr.load_state_dict(torch.load(esr_path, map_location='cpu'), strict=False)
assert not r.missing_keys, r.missing_keys[:5]
esr = esr.to(DEVICE).eval()

from transformers import CLIPModel, CLIPProcessor
clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEVICE).eval()
proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
feat = lambda o: (o if torch.is_tensor(o) else o.pooler_output).float().cpu()

from insightface.app import FaceAnalysis
det = FaceAnalysis(name='buffalo_l', allowed_modules=['detection'],
                   providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
det.prepare(ctx_id=0, det_size=DET_SIZE)
import librosa

In [ ]:
FEAT_DIR = f"{OUT_DIR}/meld_features"
os.makedirs(FEAT_DIR, exist_ok=True)
TEXT = dict(zip(U.cid, U.text))


def frames_of(path):
    cap = cv2.VideoCapture(path)
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 0
    keep = set(np.linspace(0, max(n - 1, 0), N_FRAMES).astype(int).tolist()) if n else set()
    out, i = [], 0
    while len(out) < len(keep):
        ok, fr = cap.read()
        if not ok:
            break
        if i in keep:
            out.append(fr)
        i += 1
    cap.release()
    return out


def largest_face(fr):
    faces = [f for f in det.get(fr) if f.det_score >= MIN_DET_SCORE]
    if not faces:
        return None
    x1, y1, x2, y2 = max(faces, key=lambda f: (f.bbox[2] - f.bbox[0]) * (f.bbox[3] - f.bbox[1])).bbox
    w, h = x2 - x1, y2 - y1
    if min(w, h) < MIN_FACE_PX:
        return None
    H, W = fr.shape[:2]
    x1, y1 = int(max(0, x1 - FACE_MARGIN * w)), int(max(0, y1 - FACE_MARGIN * h))
    x2, y2 = int(min(W, x2 + FACE_MARGIN * w)), int(min(H, y2 + FACE_MARGIN * h))
    return fr[y1:y2, x1:x2]


def audio_527(c):
    p = f"{WORK}/audio/{c}.wav"
    if not os.path.exists(p):
        return torch.zeros(527), False
    try:
        wav, _ = librosa.load(p, sr=AUDIO_SR, mono=True)
    except Exception:
        return torch.zeros(527), False
    if len(wav) < AUDIO_SR // 10 or np.abs(wav).max() < 1e-4:
        return torch.zeros(527), False
    L = 3 * AUDIO_SR                                   # the Hi-EF preprocessing: trim silence, centre 3 s
    wav, idx = librosa.effects.trim(wav, top_db=20)
    if len(wav) < L:
        wav = np.pad(wav, (0, L - len(wav)))
    elif len(wav) > L:
        mid = len(wav) // 2
        wav = wav[mid - L // 2: mid - L // 2 + L]
    x = torch.tensor(wav[None, None] * 32768.0, dtype=torch.float32, device=DEVICE)
    with torch.no_grad():
        return esr(x)[0].float().cpu(), True


todo = [c for c in CLIPS if not os.path.exists(f"{FEAT_DIR}/{c.replace('/', '_')}.pt")]
print(f"features to extract: {len(todo)} / {len(CLIPS)}", flush=True)
stats = {'no_video': 0, 'face_frames': 0, 'frames': 0, 'audio': 0}
t0 = time.time() if 'time' in dir() else None
import time
t0 = time.time()
for n, c in enumerate(todo):
    key = (c[:2], int(c[2:7]), int(c[8:]))
    frs = frames_of(vids[key]) if key in vids else []
    stats['no_video'] += not frs
    face = torch.zeros(N_FRAMES, 512); ori = torch.zeros(N_FRAMES, 512); fmask = torch.zeros(N_FRAMES, dtype=torch.bool)
    if frs:
        crops = [largest_face(f) for f in frs]
        imgs = [Image.fromarray(cv2.cvtColor(f, cv2.COLOR_BGR2RGB)) for f in frs]
        cimg = [Image.fromarray(cv2.cvtColor(cr, cv2.COLOR_BGR2RGB)) for cr in crops if cr is not None]
        with torch.no_grad():
            o = feat(clip.get_image_features(pixel_values=proc(images=imgs + cimg, return_tensors='pt').pixel_values.to(DEVICE)))
        ori[:len(frs)] = o[:len(frs)]
        j = len(frs)
        for k, cr in enumerate(crops):
            if cr is not None:
                face[k], fmask[k] = o[j], True
                j += 1
        stats['frames'] += len(frs); stats['face_frames'] += int(fmask.sum())
    with torch.no_grad():
        tok = proc.tokenizer([TEXT.get(c, '') or ' '], truncation=True, max_length=77, padding=True, return_tensors='pt')
        txt = feat(clip.get_text_features(**{k: v.to(DEVICE) for k, v in tok.items()}))[0]
    aud, found = audio_527(c)
    stats['audio'] += found
    torch.save({'face_features': face.half(), 'ori_features': ori.half(), 'face_valid_mask': fmask,
                'text_feature': txt.half(), 'audio_feature': aud, 'audio_found': found},
               f"{FEAT_DIR}/{c.replace('/', '_')}.pt")
    if (n + 1) % 500 == 0:
        el = (time.time() - t0) / 60
        print(f"{n + 1}/{len(todo)} | {el:.1f} min | ETA {el / (n + 1) * (len(todo) - n - 1):.0f} min | {stats}", flush=True)
print("done", stats)

## Sanity checks

In [ ]:
have = {f[:-3].replace('_', '/', 1) for f in os.listdir(FEAT_DIR) if f.endswith('.pt')}
assert set(CLIPS) <= have, f"{len(set(CLIPS) - have)} clips without features"
sample = [torch.load(f"{FEAT_DIR}/{c.replace('/', '_')}.pt") for c in CLIPS[:: max(1, len(CLIPS) // 300)]]
print(f"face-valid frame share {np.mean([s['face_valid_mask'].float().mean().item() for s in sample]):.2f} | "
      f"audio found {np.mean([s['audio_found'] for s in sample]):.2f} | "
      f"text norm {np.mean([s['text_feature'].float().norm().item() for s in sample]):.2f}")
print("outputs:", os.listdir(OUT_DIR))